# Programming for Data Science — Assignment 07
## Assignment 07: Neural Networks as Python Classes
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Assignments/Assignment_07_Neural_Network_Classes.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This assignment goes with the in-class notebook **`11_Classes_OOP.ipynb`**. First you walk through the whole roadmap of a neural-network project on a tiny network, from loading the data to a final test. Then you copy that code and edit it into a network of your own, written as a Python class, and compare it with the logistic regression you fit in Assignment 06.

About this notebook:  
The design by Chenguang Pan, in collaboration with AI. I am responsible for its accuracy.

---
### 📋 How this assignment works (Read me carefully)

- **Run on Google Colab.** Open this notebook with the badge above, then *File ▸ Save a copy in Drive* so you have your own copy to edit. PyTorch, the library we use for the networks, comes installed in Colab, so there is nothing to install.
- **The data.** The same two files as in Assignment 06: 4,424 students at a college in Portugal, split at random into a training file and a test file with 2,212 students each. This time a fifth of the training file is set aside as validation students, for checking a network while you work on it. The test file is used once, at the very end.
- **The question.** Also the same: at the end of the first semester, who is likely to drop out? Every model answers 1 ("likely to drop out") or 0 for each student.
- **What is new.** A neural network. In PyTorch every network is a Python class, so this week the classes from Session 11 go to work. You do not need any math or statistics: the notebook explains each idea in plain words, and the main number you read is accuracy, the share of students a model guesses right.
- **Three parts.** Part 1 runs the whole roadmap, all ten steps, on a tiny network. You read and run it, and answer three short questions. In Part 2 you make the network your own: you copy the Part 1 cells for steps 6 to 9 and edit them into a bigger network. Part 3 is the final check on the test students, and what you make of it.
- **AI help.** Try each part on your own first. After that you may ask an AI tool to explain an error or a concept, as the syllabus describes. Name the tool and what you used it for in `ai_note`, in the cell under Before you start. Before you ask an AI tool, download your notebook and keep that copy as your draft; the Submission section says what to upload.
- **Handle with care.** These are real, anonymized students. When a model flags someone, it means an advisor should check in.
- Use **clear names**, give your class and its `forward` method a **docstring**, and drop a **comment** on the tricky lines.

**Total: 100 points** — Part 1: 15 · Part 2: 50 · Part 3: 25 · Code quality: 10.  

Coding load is light for this assignment, but the thinking and understanding load is heavier. 

---
### ✍️ Before you start
Put your name in the next cell, and write down any AI tool you used (leave it empty if you used none). Then run the setup cell right after it.

In [ ]:
student_name = ""   # <-- type your name between the quotes
ai_note = ""        # <-- e.g. "ChatGPT: explained a shape error in Part 2" (leave "" if none)
print("Submission by:", student_name)

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch                          # <-- a famous package for modern AI
from torch import nn                  # nn holds PyTorch's building blocks for neural networks

# The same two files as in Assignment 06, read straight from the public course repo.
TRAIN_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/dropout_train.csv"
TEST_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/dropout_test.csv"

print("pandas", pd.__version__, "| PyTorch", torch.__version__)
print("Setup complete ✅")

---
## Part 1 — The roadmap, on a tiny network (read and answer the questions, no coding. 15 points)
  
**Why neural networks?**  
Neural networks (NNs) are the most flexible and powerful models in machine leanring. Theoretically, when data is abundant, a NN can learn any statistical pattern. That is why people use it for prediction or any fundamental task in modern AI, like Generverative Adversarial Networks (GANs), diffusion models, reinforment learning, or Large Language Models (LLMs).  

I intentionally design this assignment to guide you through the basic knowledge of neural networks. You do not need to understand every technical details behind the NN. The overall key takeway for this assignment is: **when you are handling complex data, say thousands of students with 10 or more bancground variables, and you want to build a model with excellent predictive performance, try NN, although it might not be the best model.**

Every network in this assignment goes through the ten steps below. Part 1 runs all ten on a tiny network, so that you see each step once before you make a network of your own. Read each cell before you run it. You do not need to change anything here; the only writing is three short answers at the end.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_roadmap.png" width="800" alt="The ten steps of the roadmap, with the three groups of students each step uses">
</p>

Figure source: Made with Claude.

### Steps 1 and 2: load and prepare

The next cell reads both files, cleans the column names with `snake_case` from Assignment 06, and keeps the same 12 predictors you could use there: every column except `student_id`, `course`, and `target`. A network needs numbers, so `gender` becomes 1 for male and 0 for female, and `daytime_evening_attendance` becomes 1 for evening classes and 0 for daytime. The outcome is 1 for a student who dropped out and 0 for everyone else.

Please read the following cell and try to understand it. You do not need to change anything here. 

In [ ]:
# Steps 1 and 2: load the two files and prepare them — run me, do not edit
def snake_case(name):
    """Return name in lowercase, with every run of other characters turned into one underscore (Assignment 06)."""
    return "_".join("".join(ch if ch.isalnum() else " " for ch in name.lower()).split())

def prepare(df):
    """Return the 12 predictors as numbers and the 0/1 dropout outcome for the students in df."""
    df = df.rename(columns={col: snake_case(col) for col in df.columns})
    X = df.drop(columns=["student_id", "course", "target"])
    X["gender"] = (X["gender"] == "male").astype(int)                                          # 1 = male
    X["daytime_evening_attendance"] = (X["daytime_evening_attendance"] == "evening").astype(int)   # 1 = evening
    y = (df["target"] == "Dropout").astype(int)                                                # 1 = dropped out
    return X, y

X_file, y_file = prepare(pd.read_csv(TRAIN_URL))                 # steps 1 and 2 for the training file
X_test_table, y_test_table = prepare(pd.read_csv(TEST_URL))      # and for the test file

print("the 12 predictors:", list(X_file.columns))

### Step 3: set aside the validation students

When we are training a NN, we need to monitor and evaluate the fitting progress to see whether it works well on an unseen dataset, this unseen dataset is called validating set. After finishing the training, we test our fitted model on the other unseen dataset, we call it testing set. Therefore, we need to seperate the observed data into three piecese: training set, validating set, and the testing set.

In [ ]:
# Step 3: hold out 20% of the training file as validation students — run me, do not edit
valid_rows = X_file.sample(frac=0.2, random_state=5001).index        # the same students for everyone
X_valid_table, y_valid_table = X_file.loc[valid_rows], y_file.loc[valid_rows]
X_train_table, y_train_table = X_file.drop(index=valid_rows), y_file.drop(index=valid_rows)

print("training students:", len(X_train_table), "| validation students:", len(X_valid_table),
      "| test students:", len(X_test_table))

### Steps 4 and 5: scale, then turn the tables into tensors

Admission grades run from about 95 to 190, while most other columns are just 0 or 1, and a network learns more smoothly when its inputs are on similar scales. So within each column, a value becomes a z-score: how far that student sits from the training students' average on that column, counted in standard deviations, where a standard deviation is the typical distance from the average. A z-score of 0 is average, and 1 is one typical distance above it.

The averages and standard deviations come from the training students only, and the validation and test students are rescaled with those same numbers. Like the norms in norm-referenced scoring, they are set once and used for everyone who comes later. It is the idea behind the `Analysis` class in the Session 11 slides. Finally the tables become tensors, PyTorch's version of a NumPy array.

In [ ]:
# Steps 4 and 5: scale with the training students' numbers, then make tensors — run me, do not edit
train_mean = X_train_table.mean()         # learned from the training students only
train_sd = X_train_table.std()

# covert the input numbers into a tensor is a critcial step for efficient computation
def to_tensor(table):
    """Return a pandas table or Series as a PyTorch tensor of decimal numbers."""
    return torch.tensor(table.to_numpy(), dtype=torch.float32)

X_train = to_tensor((X_train_table - train_mean) / train_sd)
X_valid = to_tensor((X_valid_table - train_mean) / train_sd)     # the training students' numbers,
X_test = to_tensor((X_test_table - train_mean) / train_sd)       # used for every group
y_train, y_valid, y_test = to_tensor(y_train_table), to_tensor(y_valid_table), to_tensor(y_test_table)

print("X_train:", tuple(X_train.shape), "| X_valid:", tuple(X_valid.shape), "| X_test:", tuple(X_test.shape))

### Step 6: build the network as a class

Think of a network as layers of small calculators, called units. Each unit in a hidden layer does two simple things.

1. It takes a weighted sum of the numbers that come into it, plus a constant. The weights say how much each input counts, and the network learns them from the training students.
2. It passes the sum through a bend called ReLU, which keeps positive numbers and turns negative ones into 0. Without the bend, a stack of layers would collapse into one big weighted sum.

A student's 12 numbers go into the first hidden layer. Its units hand their results to the last layer, which has a single unit and gives one risk score per student. That unit only takes the weighted sum, with no bend, so the risk score can be any number, negative or positive. Later, `torch.sigmoid` squeezes each risk score into a probability between 0 and 1 (a risk score below 0 becomes a probability below one half), and the guess is 1 ("likely to drop out") when the probability is at least 0.5, the same cutoff as in Assignment 06. So in every network you write, the last layer gets no bend and no squeeze. A network with no hidden layer is exactly the logistic regression from Assignment 06: one weighted sum, squeezed into a probability.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_anatomy.png" width="800" alt="TinyNet: 12 inputs, 8 hidden units with the ReLU bend, and one output risk score">
</p>

Figure source: Made with Claude.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_one_unit.png" width="800" alt="One hidden unit: inputs times weights, plus a constant, then the ReLU bend">
</p>

Figure source: Made with Claude.

Words you will meet in this assignment:

| Word | What it means here |
|---|---|
| unit | one small calculator: a weighted sum of its inputs plus a constant, followed by the bend in a hidden layer |
| layer | a group of units that all read the same inputs. `nn.Linear(12, 8)` is a layer that reads 12 numbers and has 8 units |
| hidden layer | any layer between the inputs and the last layer |
| weight | a number the network learns, saying how much one input counts for one unit. The weight counts in this notebook include the constants too |
| ReLU | the bend: keeps positive numbers, turns negative ones into 0 |
| risk score, probability | the last layer gives a risk score, which can be any number; `torch.sigmoid` squeezes it into a probability between 0 and 1 |
| loss | one number that says how far the network's probabilities are from what really happened; training makes it small |
| optimizer | the rule that nudges the weights to make the loss smaller; the learning rate sets the size of each nudge |
| epoch | one round in which the network looks at all the training students and nudges its weights once |
| overfitting | memorizing the training students instead of learning patterns that also hold for new students |

Here is the network from the pictures, written as a class. Read it the way you read `GradStudent` in Session 11. If we skipped section 2.9 of the class notebook (inheritance) in class, read it before you go on.

In [ ]:
# Step 6: the network as a class — run me, do not edit
class TinyNet(nn.Module):                         # <---- inherits from PyTorch's Module class
    """A small network: 12 inputs, one hidden layer of 8 units, one output risk score."""

    def __init__(self, n_inputs=12, n_hidden=8):
        super().__init__()                        # <---- let nn.Module set itself up first
        self.fc1 = nn.Linear(n_inputs, n_hidden)  # <---- the hidden layer: 12 numbers in, 8 units
        self.fc2 = nn.Linear(n_hidden, 1)         # <---- the last layer: 8 numbers in, 1 risk score out

    def forward(self, x):
        """Send a batch of students x through the layers and return one risk score per student."""
        x = torch.relu(self.fc1(x))               # <---- the hidden layer, then the bend
        return self.fc2(x)                        # <---- no bend and no squeeze after the last layer

torch.manual_seed(5001)          # the same random starting weights every time you run this cell
tiny = TinyNet()                 # <---- one object built from the class
print(tiny)                      # nn.Module's __repr__ lists the layers

Everything here comes from Session 11.

- `class TinyNet(nn.Module):` makes PyTorch's `nn.Module` the parent class, the way `GradStudent(Student)` did. `TinyNet` inherits everything a network needs, such as how to keep track of its weights and how to print itself, and adds only what makes it different: its layers and its `forward` method.
- `super().__init__()` runs the parent's setup first, as in `GradStudent`. Online you will often see the older spelling `super(TinyNet, self).__init__()`, which does the same thing.
- `self.fc1` and `self.fc2` are attributes, and each one holds an object: a layer, with its weights inside. That is composition, as when a `Course` held `Student` objects. (`fc` stands for "fully connected", a common name for this kind of layer.) `n_inputs` and `n_hidden` are settings with default values, so `TinyNet(n_hidden=32)` builds a wider version from the same class.
- `forward` says how the numbers flow from layer to layer. It overrides the `forward` of `nn.Module`, which only raises an error, so every network has to write its own. You never call it by name. Writing `tiny(X)` runs it, through a dunder method called `__call__` that `nn.Module` defines, the same way Python calls `__repr__` when you print an object.
- Layers are modules too, so they have `__call__` as well. Inside `forward`, `self.fc1(x)` sends `x` through the layer stored in `self.fc1`. That is why these attributes, unlike `titanic.shape`, are used with parentheses: they hold objects that can be called.
- `torch.manual_seed(5001)` fixes the random starting weights, so you get the same numbers every time you run the cell. A different seed, or a different version of PyTorch, gives slightly different numbers.

### Step 7: choose the loss and the optimizer

Training needs two more objects. The loss function turns the network's risk scores into one number that says how far off the guesses are. `nn.BCEWithLogitsLoss` is the usual choice for a yes/no outcome: it squeezes each risk score into a probability itself, charges little when the network is confident and right, and charges a lot when it is confident and wrong. The optimizer is the rule that nudges the weights to make the loss smaller. `torch.optim.Adam` is a popular one, and its learning rate, `lr`, sets how big each nudge is. The optimizer is built from `tiny.parameters()`, the weights inside `tiny`, so it knows which numbers it may change.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_training_loop.png" width="800" alt="One epoch of training: forward, loss, backward, step">
</p>

Figure source: Made with Claude.

In [ ]:
# Step 7: the loss function and the optimizer — run me, do not edit
loss_fn = nn.BCEWithLogitsLoss()                             # how far off the risk scores are, for a yes/no outcome
optimizer = torch.optim.Adam(tiny.parameters(), lr=0.01)     # nudges the weights inside tiny; lr = learning rate

### Step 8: train

Training repeats four steps, once per epoch, on the training students only:

1. **Forward.** The network computes a risk score for every training student.
2. **Loss.** The loss function compares those risk scores with what really happened.
3. **Backward.** `loss.backward()` works out, for every weight, whether nudging it up or down would make the loss smaller.
4. **Step.** `optimizer.step()` nudges every weight a little in that direction.

To watch the network learn, the loop writes down two accuracies every 10 epochs, one for the training students and one for the validation students. The first cell below defines two small helpers for that; the second cell is the loop itself.

In [ ]:
# Two small helpers for measuring accuracy — run me, do not edit
def guess(net, X):
    """Return a 0/1 guess per student: 1 when the network's probability of dropping out is at least 0.5."""
    net.eval()                                          # guessing mode: turns nn.Dropout (Part 2) off
    with torch.no_grad():                               # only guessing, so PyTorch skips the training bookkeeping
        risk_scores = net(X).squeeze(1)                 # one risk score per student, as a flat list
        probability = torch.sigmoid(risk_scores)        # the squeeze: any number becomes a probability between 0 and 1
    return (probability >= 0.5).float()

def accuracy(net, X, y):
    """Return the share of students whose guess matches what happened, rounded to 3 decimals."""
    return round(float((guess(net, X) == y).float().mean()), 3)

In [ ]:
# Step 8: train for 500 epochs — run me, do not edit
history = {"epoch": [], "train_accuracy": [], "valid_accuracy": []}

for epoch in range(1, 501):
    tiny.train()                                        # training mode: turns nn.Dropout on, if a network has it
    optimizer.zero_grad()                               # forget the nudges from the last epoch
    risk_scores = tiny(X_train).squeeze(1)              # 1. forward: one risk score per training student
    loss = loss_fn(risk_scores, y_train)                # 2. loss: how far off, as one number
    loss.backward()                                     # 3. backward: which way to nudge each weight
    optimizer.step()                                    # 4. step: nudge every weight a little
    if epoch % 10 == 0:                                 # every 10 epochs, write down both accuracies
        history["epoch"].append(epoch)
        history["train_accuracy"].append(accuracy(tiny, X_train, y_train))
        history["valid_accuracy"].append(accuracy(tiny, X_valid, y_valid))

print("loss after the last epoch:", round(loss.item(), 3))

### Step 9: validate

The next cell draws the two accuracies the loop wrote down. If the training line keeps climbing while the validation line stays flat or slides down, the network is memorizing the training students. That is overfitting. When you try different settings for a network, this is the step where you compare them: change a setting, go back to step 6, and train again. In Part 2 you build a bigger network and compare its plot with this one.

In [ ]:
# Step 9: compare the training and the validation students — run me, do not edit
def plot_history(history, title):
    """Draw the accuracy on the training and the validation students as training goes on."""
    if not history["epoch"]:
        print("Train for at least 10 epochs to get a plot.")
        return
    nobody = float((y_valid == 0).float().mean())      # the accuracy of guessing "nobody drops out"
    lowest = min(history["train_accuracy"] + history["valid_accuracy"])
    plt.figure(figsize=(7, 3.5))
    plt.plot(history["epoch"], history["train_accuracy"], label="training students")
    plt.plot(history["epoch"], history["valid_accuracy"], label="validation students")
    plt.axhline(nobody, color="gray", linestyle=":", label=f'"nobody drops out" ({nobody:.3f})')
    plt.ylim(min(0.6, lowest - 0.02), 1.0)              # the same scale for every plot, unless a line falls lower
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title)
    plt.legend()
    plt.show()
    if history["valid_accuracy"][-1] < nobody:
        print("⚠️ The validation accuracy ended below guessing that nobody drops out. Check that forward has no bend "
              "and no torch.sigmoid after the last layer, and that the optimizer was built from this network's parameters.")

plot_history(history, "TinyNet: one hidden layer of 8 units")

### Step 10: test once

The final check uses the test students, whom no step so far has touched. `report` collects a network's numbers in a dictionary with the same keys as `final_results` in Assignment 06, plus the number of weights and the validation accuracy. You will use it again in Part 3. Here it runs once on `tiny`, so you can see what it returns.

In [ ]:
# Step 10: the final check on the test students — run me, do not edit
def report(name, net):
    """Return a network's numbers for the final table, with the keys of final_results in Assignment 06."""
    test_guess = guess(net, X_test)
    is_dropout = y_test == 1
    return {"model": name,
            "weights": sum(p.numel() for p in net.parameters()),     # how many numbers the network learned
            "train_accuracy": accuracy(net, X_train, y_train),
            "valid_accuracy": accuracy(net, X_valid, y_valid),
            "test_accuracy": accuracy(net, X_test, y_test),
            "dropouts_caught": round(float(test_guess[is_dropout].mean()), 3)}

tiny_results = report("tiny network (8)", tiny)
print(tiny_results)

**Your turn (15 points).** Use the code cell for question 1, then answer all three questions in a sentence or two each in the text cell.
1. `tiny.fc1` is an attribute that holds a layer, and the layer has attributes of its own. Print `tiny.fc1.weight.shape` and `tiny.fc1.bias.shape` (the bias holds the constants), and the same two for `tiny.fc2`. What do the numbers mean, and how do they add up to the 113 weights in `tiny_results`?
2. In the plot from step 9, how do the two lines move as training goes on? Is there a big gap between the training and the validation students at the end?
3. Why does the roadmap keep the test students out of steps 8 and 9? What would go wrong if you kept changing a network until it looked good on the test students?

In [ ]:
# Your code here: question 1

*Double-click this cell and write your answers here.*

---
## Part 2 — Make it your own (50 points)

Now you follow the roadmap for a network of your own, by copying Part 1 and editing it. Steps 1 to 5 stay as they are: your network learns from the same `X_train` and `y_train` and is checked on the same validation students. So you only need the cells for steps 6 to 9. Copy each one into the matching cell below, then make these edits.  
(a) **Step 6, the class.** Rename `TinyNet` to `MyNet` and give it a second hidden layer, for example 64 units and then 16. The template below shows which lines change. Then build one object with `my_net = MyNet()`, right after `torch.manual_seed(5001)`.  
(b) **Step 7, the optimizer.** Build a new optimizer from your network with the same line, changed to `my_net.parameters()`. Keep the name `optimizer`, and the loop needs no other change. The loss function stays as it is.  
(c) **Step 8, the loop.** Change `tiny` to `my_net` in four places and `history` to `my_history` in four places, so that you train your network and keep TinyNet's results.  
(d) **Step 9, the plot.** Copy only the last line of the step 9 cell, and change it to plot `my_history` with a title that describes your network.  
(e) **One choice of your own.** Change one more thing: the number of units, the number of epochs, or an `nn.Dropout` layer (explained below). In a comment under your plot, say in two or three sentences what you changed and how your plot differs from TinyNet's.

Then run Checkpoint 1.

Syntax:
```python
class MyNet(nn.Module):
    """One sentence on your network's shape."""

    def __init__(self, n_inputs=12):
        super().__init__()
        self.fc1 = nn.Linear(n_inputs, units_1)    # <---- 12 numbers in, units_1 out
        self.fc2 = nn.Linear(units_1, units_2)     # <---- new: the second hidden layer takes in what fc1 gives out
        self.fc3 = nn.Linear(units_2, 1)           # <---- the last layer gives one risk score

    def forward(self, x):
        """Send a batch of students x through the layers and return one risk score per student."""
        x = torch.relu(self.fc1(x))                # <---- the bend after each hidden layer
        x = torch.relu(self.fc2(x))                # <---- new: the second hidden layer and its bend
        return self.fc3(x)                         # <---- no bend and no squeeze after the last layer
```

**A different kind of dropout.** PyTorch has a layer called `nn.Dropout`, and the name is a coincidence: it has nothing to do with students dropping out. During training, `nn.Dropout(0.5)` switches off each unit by a coin flip, with a new flip for every student in every epoch, so about half of the units sit out each time. The number is the share of units switched off. Because any unit might be missing, the network cannot lean on a few units to memorize particular students. When the network makes its guesses, `guess` turns dropout off, and every unit takes part. Create the layer once in `__init__`, and use it in `forward` right after a bend:

```python
self.dropout = nn.Dropout(0.5)                # in __init__: about half of the units sit out while training
x = self.dropout(torch.relu(self.fc1(x)))     # in forward: the bend, then the dropout layer
```

🧰 **When PyTorch complains**

| The error ends with | What it usually means |
|---|---|
| `mat1 and mat2 shapes cannot be multiplied (1770x64 and 32x16)` | two layers do not fit together: one gives out 64 numbers and the next expects 32 |
| `Target size (torch.Size([1770])) must be the same as input size (torch.Size([1770, 16]))` | the network does not end with one risk score: the last layer must be `nn.Linear(..., 1)`, and `forward` must return its output |
| `cannot assign module before Module.__init__() call` | `super().__init__()` is missing, or it comes after the first layer |
| `'NoneType' object has no attribute 'squeeze'` | `forward` has no `return` |
| `optimizer got an empty parameter list`, or `object has no attribute 'fc1'` | a layer was stored without `self.`, or `__init__` is misspelled |
| `missing 1 required positional argument: 'self'` | you used the class instead of the object: write `my_net.parameters()`, not `MyNet.parameters()` |
| `name 'units_1' is not defined` | a placeholder from the template above is still there: write the number of units |

Two mistakes give no error at all. If the validation accuracy comes out at about 0.35, the network guesses 1 for every student: remove any bend or `torch.sigmoid` after the last layer. If the accuracy hardly changes while you train, the optimizer probably still holds TinyNet's weights: build it from `my_net.parameters()`, and train `my_net`. Training also changes the object, so to start over, run your step 6 cell again; it builds a fresh `my_net`.

In [ ]:
# 🔎 Checkpoint helpers — run me, do not edit (the checkpoints below use these)
def check_history(name, history):
    """Print ✅ or ❌ for a training history: the keys from Part 1, and a validation accuracy above guessing."""
    keys = ["epoch", "train_accuracy", "valid_accuracy"]
    if not (isinstance(history, dict) and all(k in history for k in keys)):
        print(f"❌ {name} should be a dictionary with the keys {keys}, like history in Part 1")
        return False
    if not history["epoch"]:
        print(f"❌ {name} is empty: write down both accuracies every 10 epochs, as in Part 1")
        return False
    final = history["valid_accuracy"][-1]
    good = final > 0.70
    if good:
        note = ""
    elif abs(final - 0.348) < 0.03:
        note = (". A value near 0.35 means every guess is 1: remove any bend or torch.sigmoid "
                "after the last layer, then run steps 6 to 8 again")
    else:
        note = ". The network hardly learned: build the optimizer from my_net.parameters(), and train my_net"
    print("✅" if good else "❌", f"{name}: {len(history['epoch'])} records, final validation accuracy {final}{note}")
    return good

def checkpoint_1():
    """Check Part 2: the MyNet class, the object my_net, and my_history."""
    cls = globals().get("MyNet")
    if not (isinstance(cls, type) and issubclass(cls, nn.Module)):
        print("❌ write a class named MyNet that starts with: class MyNet(nn.Module):")
        return
    try:
        net = cls()
        with torch.no_grad():
            out = net(X_train[:5])
    except Exception as err:
        print(f"❌ MyNet does not work yet: {type(err).__name__}: {err}")
        return
    print("✅" if tuple(out.shape) == (5, 1) else "❌",
          f"MyNet gives one risk score per student (got shape {tuple(out.shape)}, expected (5, 1))")
    n_hidden = sum(isinstance(m, nn.Linear) for m in net.modules()) - 1
    print("✅" if n_hidden >= 2 else "❌", f"MyNet has {n_hidden} hidden layer(s)" +
          ("" if n_hidden >= 2 else "; give it a second hidden layer"))
    if any(isinstance(m, nn.Dropout) and m.p > 0 for m in net.modules()):
        net.train()                                      # in training mode, a used dropout layer changes every pass
        with torch.no_grad():
            used = not torch.equal(net(X_train[:50]), net(X_train[:50]))
        print("✅" if used else "❌", "MyNet uses its nn.Dropout layer in forward" +
              ("" if used else " (create it in __init__, then call it in forward)"))
    mine, history = globals().get("my_net"), globals().get("my_history")
    if not isinstance(mine, cls):
        print("❌ build one object from your class: my_net = MyNet()")
    elif check_history("my_history", history):
        same = abs(accuracy(mine, X_valid, y_valid) - history["valid_accuracy"][-1]) < 0.02
        print("✅" if same else "❌", "my_history comes from training my_net" +
              ("" if same else ": change tiny to my_net in the loop, and build the optimizer from my_net.parameters()"))

def checkpoint_2():
    """Check Part 3: all_results holds three dictionaries with numbers between 0 and 1."""
    results = globals().get("all_results")
    keys = ["model", "weights", "train_accuracy", "test_accuracy", "dropouts_caught"]
    if not (isinstance(results, list) and len(results) == 3 and all(isinstance(r, dict) for r in results)):
        print("❌ all_results should be a list of three dictionaries: a06_results, tiny_results, and the report for my_net")
        return
    for position, r in enumerate(results, start=1):
        problems = [k for k in keys if k not in r]
        problems += [k for k in keys[2:] if k in r and not (isinstance(r[k], (int, float)) and 0 < r[k] <= 1)]
        label = r.get("model", f"entry {position}, which has no 'model' key")
        print("❌" if problems else "✅", label, f"(check {', '.join(problems)})" if problems else "")

print("Checkpoint helpers ready ✅")

In [ ]:
# Step 6: build. Copy the step 6 cell from Part 1 here, then edit it into MyNet and my_net

In [ ]:
# Step 7: the optimizer for my_net

In [ ]:
# Step 8: train my_net and write down my_history

In [ ]:
# Step 9: plot my_history, then explain your choice in a comment

In [ ]:
# 🔎 Checkpoint 1 — run me, do not edit
checkpoint_1()

---
## Part 3 — The final check on the test students (25 points)

### 3.1 Test once (10 points)
Now, and only now, the test students come in.  
(a) Copy the numbers from `final_results` at the end of your Assignment 06 notebook into `a06_results` below. For `weights`, count the rows of the coefficient table in that model's summary, `Intercept` included, since a logistic regression learns one weight per row. If you did not finish Assignment 06, use the worked example's numbers: 4 weights, a train accuracy of 0.772, a test accuracy of 0.743, and 0.335 of the test dropouts caught. That model had no validation students, so it has no `valid_accuracy`.  
(b) Build a list named `all_results` with three dictionaries, in this order: `a06_results`, `tiny_results`, and `report("...", my_net)` with a name that describes your network. Turn the list into a DataFrame named `comparison`, and put `comparison` alone on the last line of the cell so that Colab shows it as a table. Then run Checkpoint 2.

In [ ]:
# (a) your Assignment 06 model
a06_results = {"model": "my logistic regression (A06)",
               "weights": 0,             # <-- rows in your coefficient table, Intercept included
               "train_accuracy": 0.0,    # <-- copy these three from final_results
               "test_accuracy": 0.0,
               "dropouts_caught": 0.0}

In [ ]:
# (b) all_results and the comparison table

In [ ]:
# 🔎 Checkpoint 2 — run me, do not edit
checkpoint_2()

### 3.2 What do you see? (15 points)
Answer in 4 to 6 sentences in the cell below.
1. Which of the three models does best on the test students?
2. Compare your network's training and validation accuracy, and its plot, with TinyNet's. Does your network memorize the training students more or less, and what in its design might explain that?
3. If the college asked which model to use for next year's students, which one would you recommend, and why? Think about the accuracy on the test students, the share of dropouts caught, and how easily you could explain the model to an advisor.

You may ask an AI tool to help you read the table. If you do, check one of its numbers against your own table and say that you did.

*Double-click this cell and write your answers here.*

---
## 📤 Submission

1. In Colab, run *Runtime ▸ Run all* and make sure every cell finishes **without errors**, with ✅ in both checkpoints.
2. Download your notebook (*File ▸ Download ▸ Download .ipynb*) and upload it to Canvas by **Monday, Nov 23, 11:59 pm**.
3. If you used AI, also upload the draft you saved before you used it, as the syllabus asks.

**Code quality (10 points)** covers the whole notebook: a docstring on your class and its `forward` method, clear names, a comment that names each step of your script, and comments on the tricky lines.

---
**Data credit.** Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict Students' Dropout and Academic Success* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89. Shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes for this assignment: 13 of the 36 student columns kept, plus `Target`; program names, `daytime`/`evening`, and `female`/`male` written in place of number codes; first-semester grades rounded to 2 decimals; a `student_id` added; the students split at random into two halves with nearly the same share of dropouts; values separated by commas instead of semicolons.